# Training BLIP — Image Captioning Aksara Lontara
## Model: `Salesforce/blip-image-captioning-base`

Notebook ini melakukan fine-tuning model **BLIP** untuk Aksara Lontara.

**Konfigurasi:**
- Batch size: 8
- Epochs: 20
- Learning rate: 2e-5
- Optimizer: AdamW
- Checkpoint setiap 5 epoch

> **Pastikan runtime menggunakan GPU T4** (Runtime > Change runtime type > GPU)

---
## Setup & Persiapan Dataset

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

import os, sys
try:
    os.chdir("/content")
except OSError:
    pass

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ImageCaptioning"
LOCAL_DIR = "/content/ImageCaptioning"

assert os.path.exists(DRIVE_DIR), f"ERROR: {DRIVE_DIR} tidak ditemukan!"
!rm -rf {LOCAL_DIR}
!cp -r {DRIVE_DIR} {LOCAL_DIR}
os.chdir(LOCAL_DIR)
print(f"Working directory: {os.getcwd()}")

!pip install -q transformers accelerate bitsandbytes Pillow nltk rouge-score tqdm

import torch
import numpy as np
import random
import json
import shutil
import matplotlib
matplotlib.rcParams["font.size"] = 11
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as calc_meteor
from rouge_score import rouge_scorer
import nltk
import warnings
warnings.filterwarnings("ignore")

nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")

RANDOM_SEED = 42
IMAGE_SIZE = 384
MAX_LENGTH = 128
NUM_EPOCHS = 20
LR = 2e-5

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RANDOM_SEED)

RAW_IMAGE_DIR = "dataset/images"
CAPTION_FILE = "dataset/captions.json"
OUTPUT_DIR = "dataset/processed"
EVAL_DIR = "hasil_evaluasi"
os.makedirs(EVAL_DIR, exist_ok=True)

print("Setup selesai!")

In [ ]:
print("=" * 60)
print("  PERSIAPAN DATASET")
print("=" * 60)

with open(CAPTION_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Entri caption: {len(raw_data)}")

flat_data = []
stat_single = 0
stat_multi = 0
for item in raw_data:
    img_id = item["images"]
    img_filename = f"{img_id}.png"
    karakter_list = item.get("karakter_list", [])
    jumlah = item.get("jumlah_karakter", len(karakter_list))
    if jumlah <= 1:
        stat_single += 1
    else:
        stat_multi += 1
    for caption in item["captions"]:
        flat_data.append({
            "image": img_filename,
            "caption": caption,
            "karakter_list": karakter_list,
            "jumlah_karakter": jumlah
        })

print(f"Total pasang (image, caption): {len(flat_data)}")
print(f"Gambar 1 karakter: {stat_single}")
print(f"Gambar 2 karakter: {stat_multi}")

from collections import defaultdict
seen_images = {}
for item in flat_data:
    img = item["image"]
    if img not in seen_images:
        seen_images[img] = item["jumlah_karakter"]

img_by_type = defaultdict(list)
for img, jml in seen_images.items():
    img_by_type[jml].append(img)

train_imgs, val_imgs, test_imgs = set(), set(), set()
for jml, imgs in img_by_type.items():
    random.shuffle(imgs)
    n = len(imgs)
    n_train = max(1, int(n * 0.8))
    n_val = max(1, int(n * 0.1))
    train_imgs.update(imgs[:n_train])
    val_imgs.update(imgs[n_train:n_train + n_val])
    test_imgs.update(imgs[n_train + n_val:])

train_data = [d for d in flat_data if d["image"] in train_imgs]
val_data = [d for d in flat_data if d["image"] in val_imgs]
test_data = [d for d in flat_data if d["image"] in test_imgs]

print(f"\nSplit: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    split_dir = f"{OUTPUT_DIR}/{name}"
    os.makedirs(f"{split_dir}/images", exist_ok=True)
    with open(f"{split_dir}/captions_{name}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    processed = set()
    errors = []
    for item in data:
        img = item["image"]
        if img in processed:
            continue
        processed.add(img)
        src = os.path.join(RAW_IMAGE_DIR, img)
        dst = os.path.join(OUTPUT_DIR, name, "images", img)
        if os.path.exists(src):
            try:
                im = Image.open(src).convert("RGB")
                im = im.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
                im.save(dst, format="PNG")
                Image.open(dst).verify()
            except Exception as e:
                errors.append(f"{img}: {e}")
        else:
            errors.append(f"{img}: file tidak ditemukan")
    print(f"  {name}: {len(processed)} gambar" + (f" ({len(errors)} error)" if errors else ""))

print("\nPersiapan dataset selesai!")

---
## Fine-Tuning BLIP

In [ ]:
print("=" * 60)
print("  FINE-TUNING BLIP")
print("=" * 60)

MODEL_NAME = "Salesforce/blip-image-captioning-base"
MODEL_DIR = "model/blip"
BATCH_SIZE = 8
os.makedirs(MODEL_DIR, exist_ok=True)


class CaptionDataset(Dataset):
    """Dataset untuk BLIP."""
    def __init__(self, json_file, image_dir, processor, max_length=128):
        with open(json_file, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.image_dir = image_dir
        self.processor = processor
        self.max_length = max_length
        print(f"  Dataset: {len(self.data)} pasang")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = os.path.join(self.image_dir, os.path.basename(item["image"]))
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (255, 255, 255))
        encoding = self.processor(
            images=image, text=item["caption"],
            padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze()
        pixel_values = encoding["pixel_values"].squeeze()
        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        result = {"pixel_values": pixel_values, "input_ids": input_ids, "labels": labels}
        if "attention_mask" in encoding:
            result["attention_mask"] = encoding["attention_mask"].squeeze()
        return result


# Load model
print("Memuat BLIP...")
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")

# Simpan processor
processor.save_pretrained(f"{MODEL_DIR}/best_model")

# DataLoader
train_ds = CaptionDataset(f"{OUTPUT_DIR}/train/captions_train.json", f"{OUTPUT_DIR}/train/images", processor, MAX_LENGTH)
val_ds = CaptionDataset(f"{OUTPUT_DIR}/val/captions_val.json", f"{OUTPUT_DIR}/val/images", processor, MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Training
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=total_steps)

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Train", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Val", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()
    val_loss = total_loss / len(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs(f"{MODEL_DIR}/best_model", exist_ok=True)
        model.save_pretrained(f"{MODEL_DIR}/best_model")
        print(f"  -> Best model! Val Loss: {best_val_loss:.4f}")

    if epoch % 5 == 0:
        ckpt = f"{MODEL_DIR}/checkpoint_epoch_{epoch}"
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt)

with open(f"{MODEL_DIR}/training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\nTraining selesai! Best Val Loss: {best_val_loss:.4f}")

---
## Evaluasi pada Test Set

In [ ]:
def compute_metrics(predictions, test_gambar, filter_jml=None):
    """Hitung BLEU-1..4, METEOR, ROUGE-L."""
    refs_all, hyps_all = [], []
    for img_key, pred in predictions.items():
        info = test_gambar[img_key]
        if filter_jml is not None and info["jumlah_karakter"] != filter_jml:
            continue
        refs = [c.lower().split() for c in info["captions"]]
        hyp = pred.lower().split()
        refs_all.append(refs)
        hyps_all.append(hyp)

    if not refs_all:
        return {k: 0 for k in ["BLEU-1","BLEU-2","BLEU-3","BLEU-4","METEOR","ROUGE-L"]}

    sm = SmoothingFunction().method1
    b1 = corpus_bleu(refs_all, hyps_all, weights=(1,0,0,0), smoothing_function=sm) * 100
    b2 = corpus_bleu(refs_all, hyps_all, weights=(0.5,0.5,0,0), smoothing_function=sm) * 100
    b3 = corpus_bleu(refs_all, hyps_all, weights=(0.33,0.33,0.33,0), smoothing_function=sm) * 100
    b4 = corpus_bleu(refs_all, hyps_all, weights=(0.25,0.25,0.25,0.25), smoothing_function=sm) * 100
    met = np.mean([calc_meteor(r, h) for r, h in zip(refs_all, hyps_all)]) * 100
    rsc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rg = []
    for refs, hyp in zip(refs_all, hyps_all):
        hyp_s = " ".join(hyp)
        rg.append(max(rsc.score(" ".join(r), hyp_s)["rougeL"].fmeasure for r in refs))
    rouge = np.mean(rg) * 100
    return {"BLEU-1": b1, "BLEU-2": b2, "BLEU-3": b3, "BLEU-4": b4, "METEOR": met, "ROUGE-L": rouge}


print("=" * 60)
print("  EVALUASI BLIP")
print("=" * 60)

# Load test data
with open(f"{OUTPUT_DIR}/test/captions_test.json", "r", encoding="utf-8") as f:
    test_items = json.load(f)

test_gambar = {}
for item in test_items:
    key = item["image"]
    if key not in test_gambar:
        test_gambar[key] = {"captions": [], "jumlah_karakter": item.get("jumlah_karakter", 1)}
    test_gambar[key]["captions"].append(item["caption"])

# Generate captions
model.eval()
predictions = {}
for img_key in tqdm(test_gambar, desc="Evaluasi"):
    img_path = os.path.join(f"{OUTPUT_DIR}/test/images", os.path.basename(img_key))
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception:
        continue
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4, early_stopping=True)
    predictions[img_key] = processor.decode(output[0], skip_special_tokens=True)

# Hitung metrik
results = {
    "keseluruhan": compute_metrics(predictions, test_gambar),
    "1_karakter": compute_metrics(predictions, test_gambar, filter_jml=1),
    "2_karakter": compute_metrics(predictions, test_gambar, filter_jml=2)
}

print("\n--- Hasil Evaluasi BLIP ---")
for kategori, metrik in results.items():
    print(f"\n  [{kategori}]")
    for k, v in metrik.items():
        print(f"    {k}: {v:.2f}%")

# Simpan
with open(f"{EVAL_DIR}/evaluasi_blip.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Plot training curve
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)
ax.plot(epochs_range, history["train_loss"], "o-", label="Train Loss", color="#2196F3")
ax.plot(epochs_range, history["val_loss"], "s-", label="Val Loss", color="#FF5722")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("BLIP - Training & Validation Loss", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/training_curve_blip.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nHasil disimpan: {EVAL_DIR}/evaluasi_blip.json")

---
## Simpan ke Google Drive

In [ ]:
print("\n" + "=" * 60)
print("  SIMPAN KE GOOGLE DRIVE")
print("=" * 60)

DRIVE_DIR = "/content/drive/MyDrive/ImageCaptioning"
LOCAL_DIR = "/content/ImageCaptioning"

for folder in [MODEL_DIR, EVAL_DIR]:
    src = os.path.join(LOCAL_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  {folder}/ -> Drive")

print("Selesai! Semua hasil tersimpan di Google Drive.")